# Install Libraries and Packages

In [1]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


# Import Libraries

In [2]:
import os
import glob
import re
import subprocess
from datetime import datetime
from dotenv import load_dotenv

# Creation of .env file to hide security information

#### Archivo .env con las credenciales de la base de datos
#### Por favor cree este archivo en la raíz donde tiene el actual script de python con el siguiente formato:

>DB_USER=xxxx # Cambia esto por tu usuario de PostgreSQL

>DB_HOST=xxxx # Cambia esto por la dirección de tu servidor PostgreSQL (usualmente localhost)

>DB_PORT=xxxx # Cambia esto por el puerto de tu servidor PostgreSQL (usualmente 5432 o 5433)

>DB_NAME=xxxx # Cambia esto por el nombre de tu base de datos PostgreSQL

>DB_PASS=xxxx # Cambia esto por tu contraseña de PostgreSQL

# Connect Local PostgreSQL Database

In [6]:
env_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path=env_path)

USER = os.getenv("DB_USER") # el sistema carga el usuario desde el archivo .env, no es necesario hardcodearlo
HOST = os.getenv("DB_HOST") # el sistema carga el host desde el archivo .env, no es necesario hardcodearlo
PORT = os.getenv("DB_PORT") # el sistema carga el puerto desde el archivo .env, no es necesario hardcodearlo
DB_NAME = os.getenv("DB_NAME") # el sistema carga el nombre de la base de datos desde el archivo .env, no es necesario hardcodearlo
RAW_PASS = os.getenv("DB_PASS") # el sistema carga la contraseña desde el archivo .env, no es necesario hardcodearla

if not all([USER, HOST, PORT, DB_NAME, RAW_PASS]):
    raise ValueError("Error crítico: No se pudieron importar las credenciales desde el archivo .env")

# Configure paths with raster files

In [4]:
PATH_POSTGRES_BIN = r"C:\Program Files\PostgreSQL\18\bin"
base_path = r"C:\Users\Carlos Andres\Desktop\Geovisor_Rio_Acre"

psql_abs = os.path.join(PATH_POSTGRES_BIN, "psql.exe")
raster_abs = os.path.join(PATH_POSTGRES_BIN, "raster2pgsql.exe")

if not os.path.exists(psql_abs) or not os.path.exists(raster_abs):
    raise FileNotFoundError(f"Error: No se encontraron las herramientas de PostgreSQL en: {PATH_POSTGRES_BIN}. Por favor verifica tu ruta de instalación.")

carpetas_principales = [
    "Conversion_HDF_GIS_Raster_p01",
    "Conversion_HDF_GIS_Raster_p02",
    "Conversion_HDF_GIS_Raster_p03",
    "Conversion_HDF_GIS_Raster_p04",
    "Conversion_HDF_GIS_Raster_p05"
]

mapeo_niveles = {
    "p01": 1,
    "p02": 2,
    "p03": 3,
    "p04": 4,
    "p05": 5
}

os.environ["PGPASSWORD"] = RAW_PASS

# Insert raster in geodatabase using PostGIS

In [5]:
def extraer_fecha(nombre_archivo):
    match_ts = re.search(r'(\d{2}[A-Z]{3}\d{4})_(\d{2})_(\d{2})_(\d{2})', nombre_archivo)
    if match_ts:
        try:
            str_fecha = f"{match_ts.group(1)}_{match_ts.group(2)}_{match_ts.group(3)}_{match_ts.group(4)}"
            return datetime.strptime(str_fecha, "%d%b%Y_%H_%M_%S")
        except:
            pass
            
    match_summary = re.search(r'(\d{2}[A-Z]{3}\d{4})', nombre_archivo)
    if match_summary:
        try:
            return datetime.strptime(match_summary.group(1), "%d%b%Y")
        except:
            pass
            
    return datetime(2026, 4, 19, 0, 0, 0)

try:
    for cp in carpetas_principales:
        sufijo_plan = cp.split('_')[-1]
        id_fk_nivel = mapeo_niveles.get(sufijo_plan, 1)
        
        # Inserción controlada en la tabla nivel_rios
        cmd_maestra = f'"{psql_abs}" -h {HOST} -U {USER} -d "{DB_NAME}" -c "INSERT INTO geovisor_data.nivel_rios (id_nivel_rios, nivel_rios) VALUES ({id_fk_nivel}, {id_fk_nivel}) ON CONFLICT (id_nivel_rios) DO NOTHING;"'
        subprocess.run(cmd_maestra, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        
        # --- PROCESAMIENTO DE WATER SURFACE ELEVATION (WSEL) ---
        subcarpetas_wsel = ["Max_WSEL_Summary", "WSE_Time_Series"]
        for sc in subcarpetas_wsel:
            ruta_busqueda = os.path.join(base_path, cp, sc, "*.tif")
            for ruta_tif in glob.glob(ruta_busqueda):
                nombre_archivo = os.path.basename(ruta_tif)
                fecha_timestamp = extraer_fecha(nombre_archivo).strftime("%Y-%m-%d %H:%M:%S")
                
                cmd_raster = f'"{raster_abs}" -a -s 32719 -f raster_wsel "{ruta_tif}" geovisor_data.manchas_inundacion_wsel | "{psql_abs}" -h {HOST} -U {USER} -d "{DB_NAME}"'
                subprocess.run(cmd_raster, shell=True, check=True, stdout=subprocess.DEVNULL)
                
                cmd_update = f'"{psql_abs}" -h {HOST} -U {USER} -d "{DB_NAME}" -c "UPDATE geovisor_data.manchas_inundacion_wsel SET fecha_data = \'{fecha_timestamp}\', id_nivel_rios = {id_fk_nivel} WHERE fecha_data IS NULL;"'
                subprocess.run(cmd_update, shell=True, check=True, stdout=subprocess.DEVNULL)
                
                print(f"Anexado WSEL: {nombre_archivo} -> Fecha HEC-RAS: {fecha_timestamp} -> Nivel FK: {id_fk_nivel}")

        # --- PROCESAMIENTO DE VELOCITY (VEL) ---
        subcarpetas_vel = ["Max_Vel_Summary", "VEL_Time_Series"]
        for sc in subcarpetas_vel:
            ruta_busqueda = os.path.join(base_path, cp, sc, "*.tif")
            for ruta_tif in glob.glob(ruta_busqueda):
                nombre_archivo = os.path.basename(ruta_tif)
                fecha_timestamp = extraer_fecha(nombre_archivo).strftime("%Y-%m-%d %H:%M:%S")
                
                cmd_raster = f'"{raster_abs}" -a -s 32719 -f raster_vel "{ruta_tif}" geovisor_data.manchas_inundacion_velocity | "{psql_abs}" -h {HOST} -U {USER} -d "{DB_NAME}"'
                subprocess.run(cmd_raster, shell=True, check=True, stdout=subprocess.DEVNULL)
                
                cmd_update = f'"{psql_abs}" -h {HOST} -U {USER} -d "{DB_NAME}" -c "UPDATE geovisor_data.manchas_inundacion_velocity SET fecha_data = \'{fecha_timestamp}\', id_nivel_rios = {id_fk_nivel} WHERE fecha_data IS NULL;"'
                subprocess.run(cmd_update, shell=True, check=True, stdout=subprocess.DEVNULL)
                
                print(f"Anexado VELOCITY: {nombre_archivo} -> Fecha HEC-RAS: {fecha_timestamp} -> Nivel FK: {id_fk_nivel}")

except subprocess.CalledProcessError as e:
    print(f"\nError de comando externo psql/raster2pgsql:")
    if e.stderr:
        print(e.stderr.decode('utf-8', errors='ignore'))
    else:
        print(e)
except Exception as e:
    print(f"Error durante la importación relacional masiva: {e}")

finally:
    if "PGPASSWORD" in os.environ:
        del os.environ["PGPASSWORD"]

Anexado WSEL: Summary_Max_Water_Surface_Elevation_p01.tif -> Fecha HEC-RAS: 2026-04-19 00:00:00 -> Nivel FK: 1
Anexado WSEL: WSE_01_19APR2026_00_00_00_p01.tif -> Fecha HEC-RAS: 2026-04-19 00:00:00 -> Nivel FK: 1
Anexado WSEL: WSE_02_19APR2026_00_15_00_p01.tif -> Fecha HEC-RAS: 2026-04-19 00:15:00 -> Nivel FK: 1
Anexado WSEL: WSE_03_19APR2026_00_30_00_p01.tif -> Fecha HEC-RAS: 2026-04-19 00:30:00 -> Nivel FK: 1
Anexado WSEL: WSE_04_19APR2026_00_45_00_p01.tif -> Fecha HEC-RAS: 2026-04-19 00:45:00 -> Nivel FK: 1
Anexado WSEL: WSE_05_19APR2026_01_00_00_p01.tif -> Fecha HEC-RAS: 2026-04-19 01:00:00 -> Nivel FK: 1
Anexado WSEL: WSE_06_19APR2026_01_15_00_p01.tif -> Fecha HEC-RAS: 2026-04-19 01:15:00 -> Nivel FK: 1
Anexado WSEL: WSE_07_19APR2026_01_30_00_p01.tif -> Fecha HEC-RAS: 2026-04-19 01:30:00 -> Nivel FK: 1
Anexado WSEL: WSE_08_19APR2026_01_45_00_p01.tif -> Fecha HEC-RAS: 2026-04-19 01:45:00 -> Nivel FK: 1
Anexado WSEL: WSE_09_19APR2026_02_00_00_p01.tif -> Fecha HEC-RAS: 2026-04-19 02:0